# Tiled Camera Analysis Sidecar

Run a post-run ImageAnalysis-style analyzer over camera external assets from a completed Bluesky/Tiled run. The raw run remains unchanged; analysis products are written under the day-level `analysis/` directory.

This notebook is for proving the Phase 3A sidecar path:

```text
Tiled raw run -> local camera asset fill -> analyzer -> analysis sidecar -> optional derived analysis run in Tiled
```

In [ ]:
from __future__ import annotations

from pathlib import Path

from geecs_bluesky.analysis import (
    resolve_analysis_config_dir,
    run_tiled_camera_image_analysis,
)
from image_analysis.analyzers.beam_analyzer import BeamAnalyzer
from image_analysis.config import load_camera_config

## Parameters

Select a completed Bluesky camera run and an ImageAnalysis camera configuration. This notebook uses `BeamAnalyzer` directly so the sidecar path exercises the real ImageAnalysis interface.

In [ ]:
RUN_ANALYSIS = False

EXPERIMENT = "Undulator"
YEAR = 2026
MONTH = 6
DAY = 29
SCAN_NUMBER = 1
DEVICE_NAME = "UC_Amp4_IR_input"

CAMERA_CONFIG_NAME = "Amp4Input"
ANALYSIS_CONFIG_DIR = None

# Optional controls.
DEVICE_TYPE = None
TILED_URI = None
TILED_API_KEY = None
ROOT_MAP = None
SHOT_NUMBERS = [1]  # e.g. [1, 2, 3]
INVOCATION_ID = None
FEATURE_TABLE_FORMAT = "jsonl"  # "jsonl" or "parquet"

# First prove sidecar writing. Enable publishing only after reviewing the sidecar.
EMIT_DERIVED_RUN = False
PUBLISH_TO_TILED = False

REPO_ROOT = Path.cwd().parents[1]
NOTES = "Notebook BeamAnalyzer sidecar analysis smoke test"

## BeamAnalyzer Setup

In [ ]:
if not CAMERA_CONFIG_NAME:
    raise ValueError("Set CAMERA_CONFIG_NAME before running BeamAnalyzer analysis.")

analysis_config_dir = resolve_analysis_config_dir(ANALYSIS_CONFIG_DIR)
camera_config = load_camera_config(CAMERA_CONFIG_NAME, config_dir=analysis_config_dir)
image_analyzer = BeamAnalyzer(camera_config)
analyzer_id = f"beam_analyzer_{CAMERA_CONFIG_NAME}_v1".lower()
analyzer_config = {
    "camera_config_name": CAMERA_CONFIG_NAME,
    "analysis_config_dir": str(analysis_config_dir),
    "camera_config": camera_config.model_dump(mode="json"),
}

print(type(image_analyzer).__name__, analyzer_id)
print(f"config_dir: {analysis_config_dir}")

## Run Analysis

With `EMIT_DERIVED_RUN=False`, this only writes sidecar artifacts. With `EMIT_DERIVED_RUN=True` and `PUBLISH_TO_TILED=True`, it also publishes a small derived analysis run to Tiled that links back to the raw run.

In [ ]:
metadata = None

if RUN_ANALYSIS:
    metadata = run_tiled_camera_image_analysis(
        year=YEAR,
        month=MONTH,
        day=DAY,
        scan_number=SCAN_NUMBER,
        experiment=EXPERIMENT,
        device_name=DEVICE_NAME,
        image_analyzer=image_analyzer,
        analyzer_id=analyzer_id,
        analyzer_config=analyzer_config,
        device_type=DEVICE_TYPE,
        tiled_uri=TILED_URI,
        tiled_api_key=TILED_API_KEY,
        root_map=ROOT_MAP,
        shot_numbers=SHOT_NUMBERS,
        invocation_id=INVOCATION_ID,
        feature_table_format=FEATURE_TABLE_FORMAT,
        emit_derived_run=EMIT_DERIVED_RUN,
        publish_to_tiled=PUBLISH_TO_TILED,
        retry_intervals=[],
        repo_root=REPO_ROOT,
        notes=NOTES,
    )
    print(metadata.model_dump(mode="json", exclude_none=True))
else:
    print("Set RUN_ANALYSIS = True after selecting a real run and analyzer.")

## Inspect Sidecar

In [ ]:
if metadata is not None:
    print(f"raw_run_uid: {metadata.raw_run_uid}")
    print(f"derived_run_uid: {metadata.derived_run_uid}")
    print(f"analysis_root: {metadata.analysis_root}")
    print(f"analysis_output_path: {metadata.analysis_output_path}")
    print(f"feature_table: {metadata.feature_table}")
    print(f"inputs: {len(metadata.inputs)}")
    print(f"outputs: {len(metadata.outputs)}")
else:
    print("Run the analysis cell first.")